In [1]:


# Cell 2 — Config & load
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
import time
import os
import ssl
import urllib.request

INPUT_CSV  = "processed/person.csv"
OUTPUT_CSV = "processed/person_siblings.csv"
BATCH_SIZE = 50
SLEEP_SEC  = 1.0

df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df)} rows from {INPUT_CSV}")
df.head()


Loaded 116245 rows from processed/person.csv


,person,name,citizenship,gender,birthDate,IMDb,nmNum
0,http://www.wikidata.org/entity/Q19864571,Rory Mallinson,NaN,male,1913-10-27T00:00:00Z,nm0540047,540047
1,http://www.wikidata.org/entity/Q54313724,Bobby Mallon,NaN,male,1919-04-07T00:00:00Z,nm0540066,540066
2,http://www.wikidata.org/entity/Q4964596,Brian Mallon,NaN,male,1952-05-12T00:00:00Z,nm0540067,540067
3,http://www.wikidata.org/entity/Q6196585,Jim Mallon,NaN,male,1956-03-19T00:00:00Z,nm0540075,540075
4,http://www.wikidata.org/entity/Q2617176,Boots Mallory,NaN,female,1913-10-22T00:00:00Z,nm0540096,540096


In [2]:

# Cell 3 — Helper functions
ssl_context = ssl.create_default_context()
ssl_context.check_hostname = False
ssl_context.verify_mode = ssl.CERT_NONE

def build_values_clause(uris):
    entities = " ".join(f"<{uri}>" for uri in uris)
    return f"VALUES ?person {{ {entities} }}"

def fetch_siblings_batch(uris, sparql_endpoint):
    values_clause = build_values_clause(uris)

    query = f"""
    SELECT ?person ?sibling ?siblingLabel WHERE {{
      {values_clause}
      OPTIONAL {{
        ?person wdt:P3373 ?sibling .
        ?sibling rdfs:label ?siblingLabel .
        FILTER(LANG(?siblingLabel) = "en")
      }}
    }}
    """

    sparql_endpoint.setQuery(query)
    sparql_endpoint.setReturnFormat(JSON)

    opener = urllib.request.build_opener(
        urllib.request.HTTPSHandler(context=ssl_context)
    )
    urllib.request.install_opener(opener)

    results = sparql_endpoint.query().convert()

    sibling_map = {}  # { person_uri: [ {id, name}, ... ] }
    for row in results["results"]["bindings"]:
        person_uri  = row["person"]["value"]
        sibling_uri = row["sibling"]["value"]  if "sibling"      in row else None
        sibling_name= row["siblingLabel"]["value"] if "siblingLabel" in row else None

        if sibling_uri:
            sibling_map.setdefault(person_uri, [])
            # avoid duplicates (Wikidata can return multi-label rows)
            entry = {"sibling_id": sibling_uri, "sibling_name": sibling_name}
            if entry not in sibling_map[person_uri]:
                sibling_map[person_uri].append(entry)

    return sibling_map

print("Helper functions defined ✅")

Helper functions defined ✅


In [4]:


# Cell 4 — Run batched queries
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.addCustomHttpHeader("User-Agent", "SiblingFetcher/1.0 (research project)")

all_uris     = df["person"].dropna().unique().tolist()
total_batches = (len(all_uris) + BATCH_SIZE - 1) // BATCH_SIZE
print(f"Unique person URIs to query: {len(all_uris)}")

sibling_map = {}

for i in range(0, len(all_uris), BATCH_SIZE):
    batch     = all_uris[i : i + BATCH_SIZE]
    batch_num = i // BATCH_SIZE + 1
    print(f"Batch {batch_num}/{total_batches} ({len(batch)} entities)...", end=" ")

    try:
        result = fetch_siblings_batch(batch, sparql)
        sibling_map.update(result)
        total_siblings = sum(len(v) for v in result.values())
        print(f"{len(result)} persons with siblings ({total_siblings} sibling links found)")
    except Exception as e:
        print(f"ERROR: {e}")

    time.sleep(SLEEP_SEC)

print(f"\nTotal persons with ≥1 sibling: {len(sibling_map)}")

Unique person URIs to query: 112718
Batch 1/2255 (50 entities)... 1 persons with siblings (1 sibling links found)
Batch 2/2255 (50 entities)... 1 persons with siblings (1 sibling links found)
Batch 3/2255 (50 entities)... 6 persons with siblings (6 sibling links found)
Batch 4/2255 (50 entities)... 5 persons with siblings (11 sibling links found)
Batch 5/2255 (50 entities)... 0 persons with siblings (0 sibling links found)
Batch 6/2255 (50 entities)... 2 persons with siblings (2 sibling links found)
Batch 7/2255 (50 entities)... 0 persons with siblings (0 sibling links found)
Batch 8/2255 (50 entities)... 3 persons with siblings (3 sibling links found)
Batch 9/2255 (50 entities)... 4 persons with siblings (4 sibling links found)
Batch 10/2255 (50 entities)... 5 persons with siblings (8 sibling links found)
Batch 11/2255 (50 entities)... 1 persons with siblings (3 sibling links found)
Batch 12/2255 (50 entities)... 5 persons with siblings (7 sibling links found)
Batch 13/2255 (50 entiti

In [5]:


# Cell 5 — Explode into a flat DataFrame (one row per person-sibling pair)
records = []
for person_uri, siblings in sibling_map.items():
    for s in siblings:
        # Extract QID from URI  e.g. http://www.wikidata.org/entity/Q12345 → Q12345
        person_qid  = person_uri.rsplit("/", 1)[-1]
        sibling_qid = s["sibling_id"].rsplit("/", 1)[-1]
        records.append({
            "person_uri"   : person_uri,
            "person_qid"   : person_qid,
            "sibling_uri"  : s["sibling_id"],
            "sibling_qid"  : sibling_qid,
            "sibling_name" : s["sibling_name"],
        })

siblings_df = pd.DataFrame(records, columns=[
    "person_uri", "person_qid", "sibling_uri", "sibling_qid", "sibling_name"
])

# Join back to get the queried person's name
name_map = df.set_index("person")["name"].to_dict()
siblings_df["person_name"] = siblings_df["person_uri"].map(name_map)

print(f"Total sibling pairs: {len(siblings_df)}")
print(f"Persons with ≥1 sibling: {siblings_df['person_qid'].nunique()}")
siblings_df.head(10)

# Cell 6 — Save
os.makedirs(os.path.dirname(OUTPUT_CSV) if os.path.dirname(OUTPUT_CSV) else ".", exist_ok=True)
siblings_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {OUTPUT_CSV} ({len(siblings_df)} rows) ✅")


Total sibling pairs: 9878
Persons with ≥1 sibling: 6201
Saved processed/person_siblings.csv (9878 rows) ✅


In [6]:

# Cell 7 — Optional: retry missing persons
# (persons in df who have no entry in sibling_map — they truly have no siblings
#  OR got a 429. Re-run this to fill gaps caused by rate-limit errors.)

queried = set(sibling_map.keys())
not_queried = [u for u in all_uris if u not in queried]
print(f"Persons not yet returning any result: {len(not_queried)}")
# These are persons with no P3373 sibling on Wikidata — that's expected for most.
# If you suspect 429 errors, uncomment and run the retry loop below.

# for i in range(0, len(not_queried), BATCH_SIZE):
#     batch = not_queried[i : i + BATCH_SIZE]
#     try:
#         result = fetch_siblings_batch(batch, sparql)
#         sibling_map.update(result)
#     except Exception as e:
#         print(f"ERROR: {e}")
#     time.sleep(SLEEP_SEC * 2)

Persons not yet returning any result: 106517
